# VIGOR-GS + RT-Splatting trên Colab: WayveScene101 scene_001 và 3D-FRONT-T scene_1

Notebook này chạy cho **một scene** (chọn ở ô cấu hình): baseline **RT-Splatting**, **VIGOR-GS** đầy đủ và các ablation. Mỗi biến thể có Phase II được chạy hai lần, **không** và **có** MLP hiệu chỉnh Fresnel `h_ψ` (`--optical_correction`, xem `utils/fresnel.py`):

`α = clip(F_Schlick(θ) + h_ψ(x, t, cos θ)) · glass_map`. Không có MLP thì `h_ψ = 0`, tức Fresnel thuần.

**Cần chuẩn bị** (các chỗ ⚠️ trong ô cấu hình):

1. Nhánh `viet_huynh` trên GitHub (repo công khai `iamkyledang/vigor_gs`).
2. Zip của scene trên Drive: `MyDrive/WayveScene101_scene_001.zip` hoặc `MyDrive/3D-FRONT-T_scene_1.zip`. Notebook chỉ giải nén `colmap/` (ảnh, pose, `frames.json`, `split.json`, `masks/`, `clean_image/`).
3. Runtime có GPU. Wayve ở `-r 2` là 1636×612, lớn gấp đôi VSGD; nếu thiếu VRAM thì đặt `DATA_DEVICE = 'cpu'`.

**Ảnh clean tạm thời là bản DAI** (Dereflection Any Image), vì DiffRS chưa chạy xong. Tên bộ ảnh clean nằm trong `RUN_ROOT` (`CLEAN_TAG`) và trong `notebook_config.json`, nên khi có zip mới với ảnh DiffRS thì đổi `CLEAN_TAG = 'diffrs'` và chạy lại: kết quả DAI được giữ nguyên ở thư mục cũ, không bị trộn.

| Thư mục | Biến thể | Phase I | Phase II |
|---|---|---|---|
| `rtsplatting` | Baseline RT-Splatting (`train.py`) | – | – |
| `full` / `full_mlp` | Model đầy đủ | train trên ảnh clean | đầy đủ / + `--optical_correction` |
| `no_factorization` / `_mlp` | (1) Không factorization `p·o` | `--ablate_factorization` | như full / + MLP |
| `no_separation` | (2a) Một field học ảnh gốc | `--ignore_clean_images` | không có Phase II nên không có bản MLP |
| `shared_field` / `_mlp` | (2b) Không có `G_R` riêng | dùng lại của `full` | `--shared_reflection_field` / + MLP. Không có MLP thì không còn gì để học (0 iteration); có MLP thì chỉ `h_ψ` được học |
| `no_virtual_camera` / `_mlp` | (3) Không camera ảo | dùng lại của `full` | `--ablate_virtual_camera` / + MLP |
| `no_fresnel` / `_mlp` | (4) Không Fresnel | dùng lại của `full` | `--ablate_fresnel` (một hằng số học được) / + MLP cộng lên hằng số đó |

Các bản `_mlp` dùng lại Phase I và glass plane của bản không MLP tương ứng, nên khác biệt giữa hai bản chỉ đến từ `h_ψ`. Mọi bước để lại `.done_<bước>` trên Drive: Colab ngắt thì chạy lại notebook từ đầu, các bước đã xong được bỏ qua.

Thời gian tham khảo trên VSGD 0758 (Colab, `-r 2`): Phase I ~35 phút, Phase II ~15 phút, RT-Splatting ~4,6 giờ. Một scene đủ bộ (3 Phase I, 10 Phase II, RT) mất khoảng 9 giờ, nhiều hơn với Wayve, nên thường phải chạy qua vài phiên Colab.

In [ ]:
# ⚠️ ĐIỀN / KIỂM TRA các giá trị trong ô này
REPO = 'iamkyledang/vigor_gs'   # repo công khai trên GitHub
BRANCH = 'viet_huynh'

SCENE = '3D-FRONT-T/scene_1'    # ⚠️ CHỌN: '3D-FRONT-T/scene_1' hoặc 'WayveScene101/scene_001'
PRESETS = {
    # zip trên Drive và độ phân giải -r: ảnh 3D-FRONT 960x540 chạy đầy đủ; Wayve 3271x1224 chạy -r 2 (1636x612)
    'WayveScene101/scene_001': {'zip': '/content/drive/MyDrive/WayveScene101_scene_001.zip', 'resolution': 2},
    '3D-FRONT-T/scene_1': {'zip': '/content/drive/MyDrive/3D-FRONT-T_scene_1.zip', 'resolution': 1},
}
ZIP_PATH = PRESETS[SCENE]['zip']
RESOLUTION = PRESETS[SCENE]['resolution']
CLEAN_TAG = 'dai'                # bộ ảnh clean trong zip: 'dai' (tạm thời); đổi thành 'diffrs' khi có zip mới
RUN_ROOT = f"/content/drive/MyDrive/vigor_gs_runs/{SCENE.replace('/', '_')}_{CLEAN_TAG}"
WHEEL_ROOT = '/content/drive/MyDrive/vigor_gs_runs/_wheels'   # cache build dùng chung giữa các scene

# VIGOR-GS. Đổi sau khi đã chạy thì phải dùng RUN_ROOT mới (notebook sẽ báo lỗi nếu lệch).
ITERS_I = 30_000                   # số iteration Phase I
ITERS_II = 30_000                  # số iteration Phase II
DEFORM_WIDTH, DEFORM_DEPTH = 256, 8
DATA_DEVICE = 'cuda'               # 'cpu' nếu thiếu VRAM (chậm hơn một chút)
RENDER_TRAIN = False               # True để render cả split train (chỉ để xem, metrics dùng split test)
EXTRA_I = []                       # tham số thêm cho train_interface.py
EXTRA_II = []                      # tham số thêm cho train_reflection.py (áp dụng cho cả bản có và không MLP)
GLASS_PLANE = None                 # [nx, ny, nz, d] để đặt glass plane bằng tay nếu extract_plane không fit được

# RT-Splatting, như notebook colab_vsgd_0758_rtsplatting.ipynb
RT_ITERS = 61_000
RT_INIT_UNTIL_ITER = 700
RT_NORM_LOSS_FROM_ITER = 700
RT_ENV_SCOPE = None                # None = không giới hạn; hoặc [cx, cy, cz, bán kính]
RT_XYZ_AXIS = [2, 1, 0]
RT_EXTRA = []

# Thứ tự chạy ở ô "Chạy". full và full_mlp trước để có số chính sớm; RT-Splatting (lâu nhất) cuối.
RUN_ORDER = [
    'full', 'full_mlp',
    'no_separation',
    'no_factorization', 'no_factorization_mlp',
    'shared_field', 'shared_field_mlp',
    'no_virtual_camera', 'no_virtual_camera_mlp',
    'no_fresnel', 'no_fresnel_mlp',
    'rtsplatting',
]

REPO_DIR = '/content/vigor_gs'
SCENE_DIR = f'/content/data/{SCENE}'
print('Scene:', SCENE, '| zip:', ZIP_PATH, '| -r', RESOLUTION, '| kết quả:', RUN_ROOT)

In [ ]:
import os
import subprocess

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)

from google.colab import drive

drive.mount('/content/drive')
os.makedirs(RUN_ROOT, exist_ok=True)

## Lấy code từ nhánh `viet_huynh`

In [ ]:
import os
import subprocess


def _git(*args):
    result = subprocess.run(['git', *args], capture_output=True, text=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})  # không treo chờ tài khoản
    if result.returncode:
        raise RuntimeError(f'git {args[0]} thất bại:\n{result.stderr or result.stdout}')
    return result.stdout.strip()


url = f'https://github.com/{REPO}.git'  # repo công khai, không cần token
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    _git('-C', REPO_DIR, 'fetch', '-q', url, BRANCH)
    _git('-C', REPO_DIR, 'reset', '-q', '--hard', 'FETCH_HEAD')
else:
    _git('clone', '-q', '--branch', BRANCH, '--single-branch', url, REPO_DIR)
CODE_VERSION = _git('-C', REPO_DIR, 'log', '-1', '--format=%h %s (%ci)')
print('Code:', CODE_VERSION)

## Cài thư viện và build các CUDA extension

Lần đầu mất khoảng 5–15 phút. Wheel được cache theo phiên bản torch, Python, loại GPU và nội dung `submodules/`. Notebook tìm cache ở `WHEEL_ROOT` và ở cache cũ của các notebook VSGD (`vigor_gs_runs/VSGD_0758/_wheels`), nên nếu đã chạy chúng thì chỉ mất vài giây. RT-Splatting cần thêm `nvdiffrast` và `open3d` (`render.py` import nó dù ta bỏ bước trích mesh).

In [ ]:
import glob
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys

import torch


def pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check).returncode


pip('install', '-q', 'plyfile', 'mediapy', 'ninja', 'setuptools==81.0.0')


def ensure_include(rel_path, header):
    """Some newer gcc / CUDA versions need these includes; adding them is harmless otherwise."""
    path = os.path.join(REPO_DIR, rel_path)
    with open(path) as f:
        source = f.read()
    if f'#include <{header}>' not in source:
        with open(path, 'w') as f:
            f.write(f'#include <{header}>\n' + source)


ensure_include('submodules/simple-knn/simple_knn.cu', 'cfloat')
ensure_include('submodules/diff-surfel-anych/cuda_rasterizer/rasterizer_impl.h', 'cstdint')

major, minor = torch.cuda.get_device_capability(0)
tree = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD:submodules'], capture_output=True, text=True, check=True).stdout.strip()
tag = f"torch{torch.__version__.replace('+', '-')}_py{sys.version_info.major}{sys.version_info.minor}_sm{major}{minor}_{tree}"
cached = [d for d in (os.path.join(WHEEL_ROOT, tag), os.path.join('/content/drive/MyDrive/vigor_gs_runs/VSGD_0758/_wheels', tag)) if os.path.exists(os.path.join(d, '.complete'))]
if cached:
    WHEEL_DIR = cached[0]
    print('Dùng wheel đã build:', WHEEL_DIR)
else:
    WHEEL_DIR = os.path.join(WHEEL_ROOT, tag)
    build_dir = '/content/_wheels_build'
    shutil.rmtree(build_dir, ignore_errors=True)
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
    for source in ('submodules/simple-knn', 'submodules/diff-surfel-anych', 'submodules/diff-gaussian-rasterization', 'git+https://github.com/NVlabs/nvdiffrast'):
        print('Build', source, flush=True)
        pip('wheel', '-q', '--no-build-isolation', '--no-deps', '-w', build_dir, source if source.startswith('git+') else os.path.join(REPO_DIR, source))
    os.makedirs(WHEEL_DIR, exist_ok=True)
    for wheel in glob.glob(os.path.join(build_dir, '*.whl')):
        shutil.copy(wheel, WHEEL_DIR)
    open(os.path.join(WHEEL_DIR, '.complete'), 'w').close()
pip('install', '-q', '--no-deps', '--force-reinstall', *glob.glob(os.path.join(WHEEL_DIR, '*.whl')))

for module in ('simple_knn._C', 'diff_surfel_anych', 'diff_gaussian_rasterization', 'nvdiffrast.torch', 'cv2'):
    importlib.import_module(module)

# render.py (RT-Splatting) import open3d ở đầu file. Ta không trích mesh (--skip_mesh), nên nếu không cài được
# open3d thì một module rỗng là đủ để import chạy qua.
STUB_DIR = '/content/_stubs'
if importlib.util.find_spec('open3d') is None and pip('install', '-q', 'open3d', check=False) != 0:
    os.makedirs(STUB_DIR, exist_ok=True)
    with open(os.path.join(STUB_DIR, 'open3d.py'), 'w') as f:
        f.write('# stub: open3d is only needed for mesh extraction, which this notebook skips\n')
    print('Không cài được open3d: dùng module rỗng (chỉ ảnh hưởng việc trích mesh, notebook không dùng).')
print('OK:', 'torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name(0))

## Dữ liệu: giải nén `colmap/` của scene và kiểm tra

Chỉ thư mục `colmap/` trong zip được giải nén. Ô này kiểm tra mỗi frame trong `split.json` đều có ảnh, mask và ảnh clean cùng kích thước, rồi in phương pháp tạo ảnh clean (đọc từ `colmap/clean_image/report.json`).

In [ ]:
import json
import os
import shutil
import zipfile

from PIL import Image


def extract_colmap(zip_path, scene_dir):
    """Extract only <prefix>colmap/ of the scene zip into scene_dir/colmap/."""
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Không thấy {zip_path}; kiểm tra PRESETS / SCENE ở ô cấu hình')
    stamp = f'{os.path.getsize(zip_path)}-{int(os.path.getmtime(zip_path))}'
    marker = os.path.join(scene_dir, 'colmap', '.extracted_from_zip')
    if os.path.exists(marker) and open(marker).read() == stamp:
        print('Đã giải nén trước đó, bỏ qua:', os.path.join(scene_dir, 'colmap'))
        return
    with zipfile.ZipFile(zip_path) as z:
        entries = [(info, info.filename.replace('\\', '/')) for info in z.infolist()]
        hits = [name for _, name in entries if name.endswith('colmap/frames.json')]
        if len(hits) != 1:
            raise RuntimeError(f'Cần đúng một colmap/frames.json trong {zip_path}, tìm thấy {len(hits)}: {hits[:3]}')
        prefix = hits[0][: -len('colmap/frames.json')]
        shutil.rmtree(os.path.join(scene_dir, 'colmap'), ignore_errors=True)
        count = 0
        for info, name in entries:
            if not name.startswith(prefix + 'colmap/') or name.endswith('/'):
                continue
            target = os.path.join(scene_dir, name[len(prefix):])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            count += 1
    with open(marker, 'w') as f:
        f.write(stamp)
    print(f'Đã giải nén {count} file từ "{prefix}colmap/" vào {os.path.join(scene_dir, "colmap")}')


def check_scene(scene_dir):
    """Every frame of split.json needs its COLMAP image, an undistorted mask and an undistorted clean image, all the same size."""
    colmap = os.path.join(scene_dir, 'colmap')
    with open(os.path.join(colmap, 'split.json')) as f:
        split = json.load(f)
    problems = [f'thiếu colmap/sparse/0/{name}' for name in ('cameras.bin', 'images.bin', 'points3D.bin') if not os.path.exists(os.path.join(colmap, 'sparse', '0', name))]
    sizes = set()
    for split_name in ('train', 'test'):
        for name in split[split_name]:
            stem = os.path.splitext(name)[0]
            paths = {
                'ảnh': os.path.join(colmap, 'images', name),
                'mask': os.path.join(colmap, 'masks', stem + '.png'),
                'ảnh clean': os.path.join(colmap, 'clean_image', split_name, 'images', stem + '.png'),
            }
            for kind, path in paths.items():
                if not os.path.exists(path):
                    problems.append(f'thiếu {kind}: {os.path.relpath(path, scene_dir)}')
            if name == split[split_name][0] and all(os.path.exists(p) for p in paths.values()):
                sizes |= {Image.open(p).size for p in paths.values()}
    if len(sizes) > 1:
        problems.append(f'ảnh, mask và ảnh clean khác kích thước: {sizes}')
    if problems:
        raise RuntimeError(f'{len(problems)} vấn đề với dữ liệu:\n  ' + '\n  '.join(problems[:10]))
    report = os.path.join(colmap, 'clean_image', 'report.json')
    method = json.load(open(report)).get('method', '?') if os.path.exists(report) else '(không có report.json)'
    print(f'Dữ liệu OK: {len(split["train"])} train / {len(split["test"])} test; ảnh, mask, ảnh clean đều {sizes.pop()}')
    print(f'Ảnh clean: {method}  (CLEAN_TAG = {CLEAN_TAG!r})')
    return method


extract_colmap(ZIP_PATH, SCENE_DIR)
CLEAN_METHOD = check_scene(SCENE_DIR)

## Hàm chạy pipeline

Mỗi bước gọi đúng script của repo, ghi log vào `<thư mục biến thể>/log.txt` và để lại `.done_<bước>` khi xong. Cấu hình được lưu vào `notebook_config.json` của mỗi biến thể; đổi cấu hình mà vẫn dùng `RUN_ROOT` cũ thì notebook dừng lại thay vì trộn kết quả.

Biến thể cần Phase I của biến thể khác (ví dụ `full_mlp` cần `full`) sẽ tự chạy Phase I và glass plane của biến thể đó nếu chưa có, nên có thể chạy theo thứ tự bất kỳ.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time

ENV = dict(os.environ, TQDM_MININTERVAL='60', PYTHONUNBUFFERED='1')  # tqdm in ra mỗi 60 s thay vì hàng nghìn dòng
if os.path.isdir(STUB_DIR):
    ENV['PYTHONPATH'] = STUB_DIR + os.pathsep + ENV.get('PYTHONPATH', '')

CONFIG = {
    'scene': SCENE,
    'clean_images': CLEAN_METHOD,
    'resolution': RESOLUTION,
    'iters_phase1': ITERS_I,
    'iters_phase2': ITERS_II,
    'deform': [DEFORM_WIDTH, DEFORM_DEPTH],
    'data_device': DATA_DEVICE,
    'extra_phase1': EXTRA_I,
    'extra_phase2': EXTRA_II,
    'glass_plane': GLASS_PLANE,
}
RT_CONFIG = {
    'method': 'RT-Splatting (train.py)',
    'scene': SCENE,
    'resolution': RESOLUTION,
    'iterations': RT_ITERS,
    'init_until_iter': RT_INIT_UNTIL_ITER,
    'norm_loss_from_iter': RT_NORM_LOSS_FROM_ITER,
    'env_scope': RT_ENV_SCOPE,
    'xyz_axis': RT_XYZ_AXIS if RT_ENV_SCOPE is not None else None,
    'data_device': DATA_DEVICE,
    'extra': RT_EXTRA,
}

PHASE1_FLAGS = {'full': [], 'no_factorization': ['--ablate_factorization'], 'no_separation': ['--ignore_clean_images']}
ABLATIONS = {  # key: (nhãn, Phase I lấy từ biến thể nào, cờ Phase II)
    'full': ('Model đầy đủ', 'full', []),
    'no_factorization': ('(1) Không factorization p·o', 'no_factorization', []),
    'shared_field': ('(2b) Không có G_R riêng', 'full', ['--shared_reflection_field']),
    'no_virtual_camera': ('(3) Không camera ảo', 'full', ['--ablate_virtual_camera']),
    'no_fresnel': ('(4) Không Fresnel', 'full', ['--ablate_fresnel']),
}
VARIANTS = {'rtsplatting': {'label': 'RT-Splatting (baseline)', 'mlp': None}}
for key, (label, source, flags) in ABLATIONS.items():
    VARIANTS[key] = {'label': label, 'mlp': False, 'phase1': source, 'phase2': flags}
    VARIANTS[key + '_mlp'] = {'label': label + ' + MLP Fresnel', 'mlp': True, 'phase1': source, 'phase2': flags + ['--optical_correction'], 'pair': key}
    if key == 'full':
        VARIANTS['no_separation'] = {'label': '(2a) Không tách trường phản chiếu (một field trên ảnh gốc)', 'mlp': None, 'phase1': 'no_separation', 'phase2': None}
unknown = [key for key in RUN_ORDER if key not in VARIANTS]
if unknown:
    raise ValueError(f'RUN_ORDER có tên không tồn tại: {unknown}; các tên hợp lệ: {list(VARIANTS)}')


def test_iterations(n):
    return [str(i) for i in sorted({n // 4, n // 2, n}) if i > 0]


def run(args, log_path):
    """Run one repo script, stream its output here and into log_path; raise if it fails."""
    print('$ python', ' '.join(args), flush=True)
    start = time.time()
    with open(log_path, 'a') as log:
        log.write(f'\n$ python {" ".join(args)}\n')
        log.flush()
        proc = subprocess.Popen([sys.executable, *args], cwd=REPO_DIR, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            print(line, end='', flush=True)
        code = proc.wait()
    print(f'  -> {(time.time() - start) / 60:.1f} phút', flush=True)
    if code:
        raise RuntimeError(f'{args[0]} lỗi (exit code {code}); xem {log_path}')


def is_done(m, step):
    return os.path.exists(os.path.join(m, f'.done_{step}'))


def mark_done(m, step):
    with open(os.path.join(m, f'.done_{step}'), 'w') as f:
        f.write(time.strftime('%Y-%m-%d %H:%M:%S') + '\n' + CODE_VERSION)


def step(m, name, args):
    if is_done(m, name):
        print(f'[{name}] đã xong trước đó, bỏ qua')
        return
    run(args, os.path.join(m, 'log.txt'))
    mark_done(m, name)


def prepare(key, config):
    """Create the variant folder and check it was not run with another configuration."""
    m = os.path.join(RUN_ROOT, key)
    os.makedirs(m, exist_ok=True)
    path = os.path.join(m, 'notebook_config.json')
    if os.path.exists(path):
        with open(path) as f:
            saved = json.load(f)
        if saved != json.loads(json.dumps(config)):
            raise RuntimeError(f'{m} đã được chạy với cấu hình khác:\n  cũ: {saved}\n  mới: {config}\nDùng RUN_ROOT mới (ví dụ đổi CLEAN_TAG), hoặc xoá thư mục này để chạy lại.')
    else:
        with open(path, 'w') as f:
            json.dump(config, f, indent=2)
    return m


def phase1_and_plane(key):
    """Phase I (and, except for no_separation, the glass plane) of a variant that trains its own interface field."""
    m = prepare(key, CONFIG)
    args = ['train_interface.py', '-s', SCENE_DIR, '-m', m, '-r', str(RESOLUTION), '--eval', '--data_device', DATA_DEVICE]
    args += ['--iterations', str(ITERS_I), '--deform_width', str(DEFORM_WIDTH), '--deform_depth', str(DEFORM_DEPTH), '--test_iterations', *test_iterations(ITERS_I)]
    step(m, 'phase1', args + list(EXTRA_I) + PHASE1_FLAGS[key])
    if key != 'no_separation':
        plane_args = ['extract_plane.py', '-m', m]
        if GLASS_PLANE is not None:
            plane_args += ['--glass_plane', *map(str, GLASS_PLANE)]
        step(m, 'plane', plane_args)
    return m


def reuse_phase1(m, source):
    """Copy Phase I and the glass plane of `source`; the copy then differs from it only in Phase II."""
    if is_done(m, 'phase1') and is_done(m, 'plane'):
        print(f'[phase1, plane] đã chép từ {source} trước đó, bỏ qua')
        return
    src = os.path.join(RUN_ROOT, source)
    if not (is_done(src, 'phase1') and is_done(src, 'plane')):
        print(f'[phase1, plane] {source} chưa có Phase I: chạy nó trước', flush=True)
        phase1_and_plane(source)
    shutil.copytree(os.path.join(src, 'interface'), os.path.join(m, 'interface'), dirs_exist_ok=True)
    shutil.copy(os.path.join(src, 'glass_plane.json'), os.path.join(m, 'glass_plane.json'))
    mark_done(m, 'phase1')
    mark_done(m, 'plane')
    print(f'[phase1, plane] chép từ {source}')


def run_rtsplatting():
    m = prepare('rtsplatting', RT_CONFIG)
    args = ['train.py', '-s', SCENE_DIR, '-m', m, '-r', str(RESOLUTION), '--eval', '--ignore_clean_images', '--data_device', DATA_DEVICE]
    args += ['--iterations', str(RT_ITERS), '--init_until_iter', str(RT_INIT_UNTIL_ITER), '--norm_loss_from_iter', str(RT_NORM_LOSS_FROM_ITER)]
    args += ['--test_iterations', *test_iterations(RT_ITERS)]
    if RT_ENV_SCOPE is not None:
        args += ['--env_scope_center', *map(str, RT_ENV_SCOPE[:3]), '--env_scope_radius', str(RT_ENV_SCOPE[3]), '--xyz_axis', *map(str, RT_XYZ_AXIS)]
    step(m, 'train', args + list(RT_EXTRA))
    step(m, 'render', ['render.py', '-m', m, '--skip_train', '--skip_mesh'])
    step(m, 'metrics', ['metrics.py', '-m', m])


def run_variant(key):
    variant = VARIANTS[key]
    print(f'===== {variant["label"]}  ->  {os.path.join(RUN_ROOT, key)}', flush=True)
    start = time.time()
    if key == 'rtsplatting':
        run_rtsplatting()
    elif variant['phase2'] is None:  # no_separation: one field on the original images, rendered directly
        m = phase1_and_plane(key)
        step(m, 'render', ['render_vigor.py', '-m', m, '--direct_only'] + ([] if RENDER_TRAIN else ['--skip_train']))
        step(m, 'metrics', ['metrics.py', '-m', m])
    else:
        if variant['phase1'] == key:
            m = phase1_and_plane(key)
        else:
            m = prepare(key, CONFIG)
            reuse_phase1(m, variant['phase1'])
        args = ['train_reflection.py', '-m', m, '--iterations', str(ITERS_II), '--test_iterations', *test_iterations(ITERS_II)]
        step(m, 'phase2', args + list(EXTRA_II) + variant['phase2'])
        step(m, 'render', ['render_vigor.py', '-m', m] + ([] if RENDER_TRAIN else ['--skip_train']))
        step(m, 'metrics', ['metrics.py', '-m', m])
    print(f'===== {variant["label"]}: xong ({(time.time() - start) / 60:.1f} phút)', flush=True)


print('Sẽ chạy:', ', '.join(RUN_ORDER))

## Chạy

Ô dưới chạy lần lượt mọi biến thể trong `RUN_ORDER`. Colab ngắt giữa chừng thì chạy lại notebook từ đầu: biến thể và bước nào xong rồi sẽ được bỏ qua. Muốn chạy riêng một biến thể thì dùng `run_variant('<tên>')` trong một ô mới.

In [ ]:
for key in RUN_ORDER:
    run_variant(key)

## Bảng kết quả (split test)

Mọi dòng đều được `metrics.py` chấm trên PNG sRGB so với **ảnh gốc có phản chiếu**: VIGOR-GS trên ảnh ghép `Î = (1−α)·B + α·R`, `no_separation` trên `B`, RT-Splatting trên ảnh render của nó. Cột `Masked …` nhân cả hai ảnh với mask rồi tính trên toàn ảnh, nên cao hơn PSNR thật trong vùng kính.

- `Δ PSNR so với RT` và `thắng / số view`: so từng cặp view với RT-Splatting.
- `Δ PSNR do MLP`: ở các dòng `+ MLP Fresnel`, chênh lệch trung bình theo cặp view so với bản không MLP cùng ablation, tức phần `h_ψ` thêm vào.

Bảng được lưu thành `RUN_ROOT/summary.csv`.

In [ ]:
import json
import os

import numpy as np
import pandas as pd

TABLE_ORDER = ['rtsplatting', 'full', 'full_mlp', 'no_factorization', 'no_factorization_mlp', 'no_separation',
               'shared_field', 'shared_field_mlp', 'no_virtual_camera', 'no_virtual_camera_mlp', 'no_fresnel', 'no_fresnel_mlp']


def load(key):
    folder = os.path.join(RUN_ROOT, key)
    if not os.path.exists(os.path.join(folder, 'results.json')):
        return None, None, None
    with open(os.path.join(folder, 'results.json')) as f:
        results = json.load(f)
    method = max(results, key=lambda name: int(name.rsplit('_', 1)[-1]))  # ours_<iter> / direct_<iter>: iteration lớn nhất
    with open(os.path.join(folder, 'per_view.json')) as f:
        per_view = json.load(f)[method]['PSNR']
    return method, results[method], per_view


def paired(views, reference):
    common = sorted(set(views) & set(reference))
    deltas = np.array([views[name] - reference[name] for name in common])
    return deltas.mean(), f'{int((deltas > 0).sum())}/{len(common)}'


loaded = {key: load(key) for key in TABLE_ORDER}
baseline_views = loaded['rtsplatting'][2]
rows = []
for key in TABLE_ORDER:
    variant = VARIANTS[key]
    method, scores, views = loaded[key]
    row = {'biến thể': variant['label'], 'MLP Fresnel': {None: '–', False: 'không', True: 'có'}[variant['mlp']], 'thư mục': key}
    if method is None:
        rows.append({**row, 'method': '(chưa chạy)'})
        continue
    row.update({'method': method, **scores})
    if baseline_views is not None and key != 'rtsplatting':
        row['Δ PSNR so với RT'], row['thắng / số view'] = paired(views, baseline_views)
    if variant.get('pair') and loaded[variant['pair']][2] is not None:
        row['Δ PSNR do MLP'], row['MLP thắng / số view'] = paired(views, loaded[variant['pair']][2])
    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RUN_ROOT, 'summary.csv'), index=False)
print(f'{SCENE}, ảnh clean: {CLEAN_METHOD}')
display(summary.round(4))

## Xem một frame test

Hàng trên: ảnh gốc, RT-Splatting, VIGOR-GS `full` và `full_mlp`. Hàng dưới: lớp truyền qua và lớp phản xạ của RT-Splatting, `B` (direct) và `α·R` của `full` và `full_mlp`. Đây chỉ là so sánh định tính: các lớp của hai phương pháp không được định nghĩa giống hệt nhau.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
from PIL import Image

FRAME = '00000.png'  # chỉ số frame trong split test


def method_dir(key):
    found = sorted(glob.glob(os.path.join(RUN_ROOT, key, 'test', '*', 'renders')), key=lambda p: int(os.path.basename(os.path.dirname(p)).rsplit('_', 1)[-1]))
    return os.path.dirname(found[-1]) if found else None


rt, full, full_mlp = method_dir('rtsplatting'), method_dir('full'), method_dir('full_mlp')
gt = next((d for d in (rt, full, full_mlp) if d), None)
panels = [
    ('Ảnh gốc (gt)', gt and os.path.join(gt, 'gt', FRAME)),
    ('RT-Splatting: render', rt and os.path.join(rt, 'renders', FRAME)),
    ('full: render', full and os.path.join(full, 'renders', FRAME)),
    ('full_mlp: render', full_mlp and os.path.join(full_mlp, 'renders', FRAME)),
    ('RT-Splatting: transmitted', rt and os.path.join(rt, 'vis', 'transmitted_' + FRAME)),
    ('RT-Splatting: specular', rt and os.path.join(rt, 'vis', 'specular_' + FRAME)),
    ('full: B (direct)', full and os.path.join(full, 'vis', 'direct_' + FRAME)),
    ('full: α·R', full and os.path.join(full, 'vis', 'reflection_contribution_' + FRAME)),
    ('full_mlp: B (direct)', full_mlp and os.path.join(full_mlp, 'vis', 'direct_' + FRAME)),
    ('full_mlp: α·R', full_mlp and os.path.join(full_mlp, 'vis', 'reflection_contribution_' + FRAME)),
]
panels = [(t, p) for t, p in panels if p and os.path.exists(p)]
if panels:
    cols = 4
    rows_ = (len(panels) + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(5 * cols, 3.2 * rows_), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (title, path) in zip(axes.ravel(), panels):
        ax.imshow(Image.open(path))
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print('Chưa có ảnh render nào.')